https://www.unav.edu/web/instituto-de-ciencia-de-los-datos-e-inteligencia-artificial<br>

Autor: Darian Horacio Grass Boada

### ¿Qué es una Red Convolutiva de Grafos?

- Una arquitectura de red neuronal muy potente para el aprendizaje automático en grafos.
- Veamos un ejemplo básico y entendamos el funamento matemático subyacente con un ejemplo simple


Formalmente, una red convolutiva de grafos (GCN) es una red neuronal que opera en grafos. Dado un grafo $G = (V, E)$, una GCN toma como entrada
- una matriz de características de entrada $N \times F^{0}$, $X$, donde $N$ es el número de nodos y $F^{0}$ es el número de características de entrada para cada nodo, y
- una representación matricial $N \times N$ de la estructura del grafo, como la matriz de adyacencia $A$ de $G$

Por lo tanto, una capa oculta en la GCN se puede escribir como $H^{i} = f(H^{i-1},A)$ donde $H^{0} = X$ y $f$ es una regla de propagación
- Cada capa $H^{i}$ corresponde a una matriz de características $N \times F^{i}$ donde cada fila es una representación de características de un nodo.
- En cada capa, estas características se agregan para formar las características de la siguiente capa utilizando la regla de propagación $f$.

### Una Regla de Propagación Simple

Una de las reglas de propagación más simples posibles es:

$$f(H^{i}, A) = \sigma(AH^{i}W^{i}) $$

donde $W^{i}$ es la matriz de pesos para la capa $i$ y $\sigma$ es una función de activación no lineal como la función ReLU. La matriz de pesos tiene dimensiones $F^{i} \times F^{i+1}$; en otras palabras, el tamaño de la segunda dimensión de la matriz de pesos determina el número de características en la siguiente capa.


##### representación de matriz de adyacencia numpy

In [ ]:
##### representación de matriz de adyacencia numpy
import numpy as np
A = np.matrix([
    [0, 1, 0, 0],
    [0, 0, 1, 1],
    [0, 1, 0, 0],
    [1, 0, 1, 0]],
    dtype=float
)
A

In [ ]:
# Comprobar si la matriz es simétrica
def es_dirigido(A):
    return not np.array_equal(A, A.T)

if es_dirigido(A):
    print("El grafo es dirigido.")
else:
    print("El grafo no es dirigido.")

##### ¡Necesitamos características! Generamos 2 características enteras para cada nodo basadas en su índice.

In [ ]:
X = np.matrix([[i, -i] for i in range(A.shape[0])], dtype=float)
X

##### Aplicando la Regla de Propagación

In [ ]:
A*X

- ¿Qué pasó? ¡La representación de cada nodo (cada fila) es ahora una suma de las características de sus vecinos! En otras palabras, la capa convolutiva del grafo representa cada nodo como un agregado de su vecindario.

- Te animo a que compruebes el cálculo por ti mismo.

### Problemas ###
1. ¡La representación agregada de un nodo no incluye sus propias características!
2. Los nodos con grados grandes tendrán valores grandes en su representación de características, mientras que los nodos con grados pequeños tendrán valores pequeños.

### Problema 1: solución ###
##### Añadiendo Bucles Propios: En la práctica, esto se hace añadiendo la matriz identidad $I$ a la matriz de adyacencia $A$ antes de aplicar la regla de propagación.

In [ ]:
I = np.matrix(np.eye(A.shape[0]))
I

In [ ]:
A_hat = A + I
A_hat * X

In [ ]:
A*X

- ¿Qué pasó con $v_{0}$?

### Problema 2: solución ###
##### Las representaciones de características pueden ser normalizadas por el grado del nodo transformando la matriz de adyacencia $A$ multiplicándola con la matriz de grado inversa $D$
$f(X,A)=D^{-1}AX$

##### Primero calculamos la matriz de grados.

In [ ]:
D = np.array(np.sum(A, axis=0))[0]
D = np.matrix(np.diag(D))
D

In [ ]:
A

##### veamos qué le sucede a la matriz de adyacencia después de transformarla.

In [ ]:
D**-1 * A

- Observa que los valores en cada fila de la matriz de adyacencia han sido divididos por el grado del nodo correspondiente a la fila

##### Aplicamos la regla de propagación con la matriz de adyacencia transformada

In [ ]:
D**-1 * A * X

- representaciones de nodos correspondientes a la media de las características de los nodos vecinos.
- esto se debe a que los pesos en la matriz de adyacencia (transformada) corresponden a pesos en una suma ponderada de las características de los nodos vecinos.

### Juntándolo Todo

> Ahora combinamos los consejos de bucle propio y normalización. Además, reintroduciremos los pesos y la función de activación

In [ ]:
D_hat = np.array(np.sum(A_hat, axis=0))[0]
D_hat = np.matrix(np.diag(D_hat))
D_hat

- Añadiendo de nuevo los Pesos
- La primera tarea es aplicar los pesos.
- Nota que aquí D_hat es la matriz de grados de A_hat = A + I, es decir, la matriz de grados de A con bucles propios forzados.

In [ ]:
W = np.matrix([[1, -1], [-1, 1]])
D_hat**-1 * A_hat * X * W

> si queremos reducir la dimensionalidad de las representaciones de características de salida, podemos reducir el tamaño de la matriz de pesos $W$

In [ ]:
W = np.matrix([[1], [-1]])
D_hat**-1 * A_hat * X * W

In [ ]:
def relu(x):
    return (np.maximum(0.0, x))


- Añadiendo una Función de Activación
- Elegimos preservar la dimensionalidad de las representaciones de características y aplicar la función de activación ReLU.

In [ ]:
W = np.matrix([[1, -1], [-1, 1]])
relu(D_hat**-1 * A_hat * X * W)

> Hecho: ¡Una capa oculta completa con matriz de adyacencia, características de entrada, pesos y función de activación!

## Zachary’s Karate Club
### Construyendo la GCN

In [ ]:
import networkx as nx

zkc = nx.karate_club_graph()
order = sorted(list(zkc.nodes()))

bb = nx.edge_betweenness_centrality(zkc, normalized=False)
nx.set_edge_attributes(zkc, bb, "weight")
zkc[0][1]["weight"]

In [ ]:
A = nx.to_numpy_array(zkc, nodelist=order, weight='None')
I = np.eye(zkc.number_of_nodes())
A_hat = A + I
D_hat = np.array(np.sum(A_hat, axis=0))
D_hat = np.matrix(np.diag(D_hat))
D_hat

In [ ]:
W_1 = np.random.normal(
    loc=0, scale=1, size=(zkc.number_of_nodes(), 4))
W_2 = np.random.normal(
    loc=0, size=(W_1.shape[1], 2))

In [ ]:
def gcn_layer(A_hat, D_hat, X, W):
    return relu(D_hat**-1 * A_hat * X * W)
H_1 = gcn_layer(A_hat, D_hat, I, W_1)
H_2 = gcn_layer(A_hat, D_hat, H_1, W_2)
output = H_2
output

In [ ]:
feature_representations = {
    node: np.array(output)[node]
    for node in zkc.nodes()}
feature_representations

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

X = []
Y = []
C = []
Id = []
for item in feature_representations:
    X.append(feature_representations[item][0])
    Y.append(feature_representations[item][1])
    C.append(zkc.nodes[item]["club"])
    Id.append(item)


#create DataFrame
df = pd.DataFrame({'x': X,
                   'y': Y,
                   'id': Id,
                   'z': C
                   })

df

In [ ]:
groups = df.groupby('z')
for name, group in groups:
    plt.plot(group.x, group.y, marker='o', linestyle='', markersize=12, label=name)
    # plt.text(group.x, group.y, group.id)

plt.legend()